**Data Collection**      
***Phenomenon of Culturomics on case of Michael Jackson Wikipedia page***

This notebook documents how the two datasets used in this project were collected:
pageviews and revision history for the Wikipedia article "Michael Jackson".     

**Step 1 Import libraries**

This stepload the Python libraries needed for the rest of the notebook.

In [ ]:
import os
import time
import random
import gzip
import urllib3
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta, UTC
from dotenv import load_dotenv


# Live collection needs a Wikipedia Bot Password 
RUN_LIVE_COLLECTION = False


**Step 2 Pageviews collection function (2009 archive)**

This step define the function that streams hourly pageview archive
files for June 2009 directly from Wikimedia, without saving them to disk.

The Goal is to show the exact method used to reconstruct pageviews for a period
before the official Pageviews API existed.

The code must be able to answer to question:    
How many times was the article "Michael Jackson" viewed on a
given hour of a given day in 2009?

he second step of the pipeline is a function that returns one number (views) for one hour, with automatic retries on network errors.


In [ ]:
HEADERS = {
    "User-Agent": (
        "DigitalHumanitiesProject/1.0 "
        "( Wikipedia account: Aim.btlv; "
        "culturomics research project)"
    )
}
ARTICLE = "Michael_Jackson"
SLEEP_BETWEEN_REQUESTS = 3.0
MAX_RETRIES = 5


def request_with_backoff(method: str, url: str, **kwargs) -> requests.Response:
    """
    Send an HTTP request and retry it if the server is rate-limiting us ,
    overloaded, or the connection times out.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.request(method, url, headers=HEADERS, timeout=60, **kwargs)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            wait_time = (attempt + 1) * 10 + random.uniform(0, 3)
            print(f"Network error ({exc}). Waiting {wait_time:.1f}s before retry...")
            time.sleep(wait_time)
            continue

        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 30))
            time.sleep(retry_after)
            continue
        if response.status_code == 503:
            time.sleep((attempt + 1) * 10 + random.uniform(0, 3))
            continue
        if response.status_code == 404:
            return response

        response.raise_for_status()
        return response

    raise RuntimeError(f"Max retries exceeded while requesting {url}")


def get_article_count_streaming(date_str: str, hour: str, article: str, lang: str = "en") -> int:
    """
    Stream one hourly pagecounts-raw archive file directly from the network,
    decompress it on the fly, and return the view count for one article.
    The file is never saved to disk. If the download is interrupted, the
    attempt is retried up to MAX_RETRIES times before giving up on that hour.
    """
    year, month = date_str[:4], date_str[4:6]
    filename = f"pagecounts-{date_str}-{hour}.gz"
    url = f"https://dumps.wikimedia.org/other/pagecounts-raw/{year}/{year}-{month}/{filename}"
    target_prefix = f"{lang} {article} "

    for attempt in range(MAX_RETRIES):
        response = request_with_backoff("GET", url, stream=True)
        if response.status_code == 404:
            return 0
        try:
            with gzip.GzipFile(fileobj=response.raw) as gz:
                for raw_line in gz:
                    line = raw_line.decode("utf-8", errors="ignore")
                    if line.startswith(target_prefix):
                        time.sleep(SLEEP_BETWEEN_REQUESTS)
                        return int(line.split()[2])
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            return 0
        except (OSError, gzip.BadGzipFile, requests.exceptions.RequestException,
                urllib3.exceptions.HTTPError) as exc:
            time.sleep((attempt + 1) * 10 + random.uniform(0, 3))
            continue
        finally:
            response.close()

    return 0


**Step 3 Pageviews collection function (2015-present API) and full run**

This step define the function that fetches daily pageviews from the
official REST API, and the function that runs the full collection: the 2009
archive days plus the 2015-present daily series.

The Goal is to cover the modern period with a single, lightweight, official API
call 

The code must be able to answer to question:    
What were the daily pageviews for the article every day since
July 2015?

The third step of the pipeline is a combined DataFrame covering both periods, saved to CSV only when `RUN_LIVE_COLLECTION` is True.


In [ ]:
def collect_2009_range(start="2009-06-20", end="2009-07-10") -> pd.DataFrame:
    """Collect daily pageviews for every day in the given 2009 range."""
    hours = [f"{h:02d}0000" for h in range(24)]
    records = []
    for date_str in [d.strftime("%Y%m%d") for d in pd.date_range(start, end, freq="D")]:
        daily_total = sum(get_article_count_streaming(date_str, h, ARTICLE) for h in hours)
        records.append({"date": pd.to_datetime(date_str, format="%Y%m%d"), "views": daily_total})
        print(f"Day {date_str} done: {daily_total} views")
    return pd.DataFrame(records)


def collect_modern_pageviews(start: str = "2015070100") -> pd.DataFrame:
    """Fetch daily pageviews from the official Wikimedia REST API, up to yesterday."""
    end = (datetime.now(UTC) - timedelta(days=1)).strftime("%Y%m%d00")
    url = (
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia/all-access/user/{ARTICLE}/daily/{start}/{end}"
    )
    response = request_with_backoff("GET", url)
    data = response.json()["items"]
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["timestamp"], format="%Y%m%d%H")
    return df[["date", "views"]]


if RUN_LIVE_COLLECTION:
    df_2009 = collect_2009_range()
    df_modern = collect_modern_pageviews()
    pageviews = pd.concat([df_2009, df_modern], ignore_index=True)
    pageviews.to_csv("michael_jackson_pageviews.csv", index=False)
else:
    pageviews = pd.read_csv("michael_jackson_pageviews.csv", parse_dates=["date"])

pageviews = pageviews.sort_values("date").reset_index(drop=True)
pageviews.head()


**Step 4  Check the pageviews data**

This step run basic checks on the collected or loaded file.

The Goal is to make sure the collection process worked and the data can be trusted.

The code must be able to answer to question:     
Are there missing values, duplicate dates, or gaps in the daily
sequence?

 The fourth step of the pipeline is a short summary confirming the data is complete, or flagging problems.


In [ ]:
print("Rows:", len(pageviews))
print("Missing values:\n", pageviews.isnull().sum())
print("Duplicate dates:", pageviews["date"].duplicated().sum())
print("Date range:", pageviews["date"].min(), "->", pageviews["date"].max())

modern = pageviews[pageviews["date"] >= "2015-07-01"]
full_range = pd.date_range(modern["date"].min(), modern["date"].max(), freq="D")
missing_days = full_range.difference(modern["date"])
print("Missing days in the 2015-present daily series:", len(missing_days))


**Step 5  Revision history collection functions**

This step define the functions used to log in with a Wikipedia Bot
Password and download the full revision history of the article.

The Goal is to show the exact method used to reconstruct edit timestamps and
article size over time.

The code must be able to answer to question:    
What was the timestamp and byte size of every revision of the
article, from the first edit to the most recent one?

The fifth step of the pipeline is functions that, together, return the complete revision history as a DataFrame. Login credentials are read from a local `.env` file and are never written in this notebook.


In [ ]:
load_dotenv()
BOT_USERNAME = os.getenv("WIKI_BOT_USERNAME")
BOT_PASSWORD = os.getenv("WIKI_BOT_PASSWORD")
API_URL = "https://en.wikipedia.org/w/api.php"
ARTICLE_TITLE = "Michael Jackson"
CHECKPOINT_FILE = "revisions_partial.csv"


def safe_json_response(response: requests.Response) -> dict:
    """Raise a clear error if the response is not valid JSON, instead of failing silently."""
    if "application/json" not in response.headers.get("Content-Type", ""):
        raise RuntimeError(f"Non-JSON response (status {response.status_code}): {response.text[:200]!r}")
    return response.json()


def login(session: requests.Session, username: str, password: str) -> None:
    """Log in using the legacy login flow, recommended for Bot Passwords."""
    if not username or not password:
        raise RuntimeError("Missing WIKI_BOT_USERNAME / WIKI_BOT_PASSWORD in .env")
    token_resp = session.get(
        API_URL,
        params={"action": "query", "meta": "tokens", "type": "login", "format": "json"},
        headers=HEADERS,
    )
    token = safe_json_response(token_resp)["query"]["tokens"]["logintoken"]
    time.sleep(1.0)
    login_resp = session.post(
        API_URL,
        data={"action": "login", "lgname": username, "lgpassword": password, "lgtoken": token, "format": "json"},
        headers=HEADERS,
    )
    if safe_json_response(login_resp).get("login", {}).get("result") != "Success":
        raise RuntimeError("Login failed")


def collect_all_revisions(session: requests.Session) -> pd.DataFrame:
    """
    Page through the full revision history in batches of 500, saving a
    checkpoint file after every batch so an interruption never loses progress.
    """
    all_revisions, rvcontinue = [], None
    while True:
        params = {
            "action": "query", "format": "json", "prop": "revisions", "titles": ARTICLE_TITLE,
            "rvprop": "timestamp|size|ids", "rvlimit": "500", "rvdir": "newer", "maxlag": "5",
            "formatversion": "2",
        }
        if rvcontinue:
            params["rvcontinue"] = rvcontinue
        resp = session.get(API_URL, params=params, headers=HEADERS)
        data = safe_json_response(resp)
        page = data["query"]["pages"][0]
        all_revisions.extend(page.get("revisions", []))
        pd.DataFrame(all_revisions).to_csv(CHECKPOINT_FILE, index=False)
        print(f"Collected {len(all_revisions)} revisions so far...")
        if "continue" in data:
            rvcontinue = data["continue"]["rvcontinue"]
            time.sleep(2.5)
        else:
            break
    df = pd.DataFrame(all_revisions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df[["timestamp", "size", "revid"]]


**Step 6  Run revision collection or load the finished file**

This step either log in and run the full collection, or load the
already-collected file, depending on `RUN_LIVE_COLLECTION`.

The Goal is to keep this notebook safe to run by default, while still keeping the
full, real collection logic visible and runnable on demand.

The code must be able to answer to question:     
What does the final revisions table look like?

The sixth step of the pipeline is the full revisions DataFrame, ready for the checks in Step 7.


In [ ]:
if RUN_LIVE_COLLECTION:
    session = requests.Session()
    login(session, BOT_USERNAME, BOT_PASSWORD)
    revisions = collect_all_revisions(session)
    revisions.to_csv("michael_jackson_revisions.csv", index=False)
else:
    revisions = pd.read_csv("michael_jackson_revisions.csv", parse_dates=["timestamp"])

revisions = revisions.sort_values("timestamp").reset_index(drop=True)
revisions.head()


**Step 7 Check the revisions data**

This step run basic checks on the collected or loaded file.

The Goal is to make sure the collection process worked and the data can be trusted.

The code must be able to answer to question:      
Are there missing values, duplicate revisions, or an incomplete
date range?

The seventh step of the pipeline a short summary confirming the data is complete, or flagging problems.


In [ ]:
print("Rows:", len(revisions))
print("Missing values:\n", revisions.isnull().sum())
print("Duplicate revision IDs:", revisions["revid"].duplicated().sum())
print("Date range:", revisions["timestamp"].min(), "->", revisions["timestamp"].max())
print("Is sorted by time:", revisions["timestamp"].is_monotonic_increasing)


**Step 8 Save a clean working copy**

This step save both checked datasets under simple, stable file names for
use in the Analysis notebook.

The Goal is to keep data collection and data analysis as two separate, independent
steps.

The code must be able to answer to question:      
What exactly will the Analysis notebook load?

The eighth step of the pipeline is two CSV files, unchanged in content, confirmed clean.


In [ ]:
pageviews.to_csv("pageviews_clean.csv", index=False)
revisions.to_csv("revisions_clean.csv", index=False)
print("Saved: pageviews_clean.csv, revisions_clean.csv")
